# 13 — P2: PGD L∞ Attack for ViT-Tiny

**Plan 2 — Phase 5.** Port of the existing MNIST-MLP PGD attack to the ViT-Tiny
architecture. The hybrid verifier (Phase 6) calls this between IBP and MILP:
if PGD succeeds, the sample is *falsified* with a concrete counterexample
(no MILP solve needed).

## Spec (Plan 2 §5)
- 100 PGD steps
- step size α = 0.1·ε
- 5 random restarts
- sign-of-gradient L∞ projection
- input clamped to `[0, 1]`

In [1]:
!pip install -q numpy torch torchvision tqdm pyyaml

In [2]:
from __future__ import annotations
import json, time, warnings
from pathlib import Path
from typing import Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as transforms
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1234); np.random.seed(1234)
print(f'Device: {device}')

Device: cuda


In [3]:
# ── ViT-Tiny class (matches notebooks 09–12) ──────────────────────────────
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.dim, self.eps = dim, eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, embed_dim=64):
        super().__init__(); self.img_size, self.patch_size = img_size, patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim, self.num_heads = embed_dim, num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=True)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=True)
    def forward(self, x):
        B, N, C = x.shape; H, D = self.num_heads, self.head_dim
        q = self.W_q(x).view(B, N, H, D).transpose(1, 2)
        k = self.W_k(x).view(B, N, H, D).transpose(1, 2)
        v = self.W_v(x).view(B, N, H, D).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.W_o(out)

class MLPBlock(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=2):
        super().__init__(); h = embed_dim * mlp_ratio
        self.fc1, self.fc2 = nn.Linear(embed_dim, h), nn.Linear(h, embed_dim)
        self.act = nn.ReLU()
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.norm1 = RMSNorm(embed_dim, eps_rms); self.attn = MHSA(embed_dim, num_heads)
        self.norm2 = RMSNorm(embed_dim, eps_rms); self.mlp  = MLPBlock(embed_dim, mlp_ratio)
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); x = x + self.mlp(self.norm2(x)); return x

class ViTTiny(nn.Module):
    def __init__(self, img_size=28, patch_size=4, in_channels=1, num_classes=10,
                 embed_dim=64, num_heads=2, num_layers=2, mlp_ratio=2, eps_rms=1e-6):
        super().__init__()
        self.cfg = dict(img_size=img_size, patch_size=patch_size,
                        in_channels=in_channels, num_classes=num_classes,
                        embed_dim=embed_dim, num_heads=num_heads,
                        num_layers=num_layers, mlp_ratio=mlp_ratio, eps_rms=eps_rms)
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        self.pos_embed   = nn.Parameter(torch.zeros(1, self.patch_embed.n_patches, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_ratio, eps_rms)
                                     for _ in range(num_layers)])
        self.norm = RMSNorm(embed_dim, eps_rms)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

In [4]:
# ── PGD L∞ attack ─────────────────────────────────────────────────────────
def pgd_linf(model: nn.Module, x: torch.Tensor, y: torch.Tensor,
             eps: float, n_steps: int = 100, n_restarts: int = 5,
             alpha: Optional[float] = None,
             clamp: tuple = (0.0, 1.0)) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Multi-restart PGD-L∞.  Returns (best_adv, success_mask).

    success_mask[i] is True if model(best_adv[i]).argmax() ≠ y[i].
    Step size defaults to α = 0.1·ε per Plan 2 §5.
    """
    if alpha is None: alpha = 0.1 * eps
    model.eval()
    lo, hi = clamp

    # Initialise best with the clean input — if the clean prediction is wrong,
    # it's already a "successful attack" (zero perturbation).
    best_loss = torch.full((len(x),), -1e9, device=x.device)
    best_adv  = x.clone()

    for r in range(n_restarts):
        if r == 0:
            delta = torch.zeros_like(x)
        else:
            delta = torch.empty_like(x).uniform_(-eps, eps)
        delta.requires_grad_(True)

        for _ in range(n_steps):
            adv  = (x + delta).clamp(lo, hi)
            loss = F.cross_entropy(model(adv), y, reduction='sum')
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
                delta = ((x + delta).clamp(lo, hi) - x).detach()
            delta.requires_grad_(True)

        with torch.no_grad():
            adv = (x + delta).clamp(lo, hi)
            per_sample_loss = F.cross_entropy(model(adv), y, reduction='none')
            mask = per_sample_loss > best_loss
            best_loss[mask] = per_sample_loss[mask]
            best_adv[mask]  = adv[mask]

    with torch.no_grad():
        success = model(best_adv).argmax(1) != y
    return best_adv, success

In [5]:
# ── Sweep PGD over ε on both models ───────────────────────────────────────
def load_vit(ckpt_path: Path) -> ViTTiny:
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = ViTTiny(**payload['cfg']).to(device)
    sd = payload.get('state_dict_materialized', payload['state_dict'])
    sd = {k: v for k, v in sd.items() if 'parametrizations' not in k}
    model.load_state_dict(sd, strict=False); model.eval()
    return model

def eval_pgd(model, loader, eps, n_steps=100, n_restarts=5):
    n = correct = robust = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        adv, _ = pgd_linf(model, x, y, eps, n_steps=n_steps, n_restarts=n_restarts)
        with torch.no_grad():
            clean_pred = model(x).argmax(1)
            adv_pred   = model(adv).argmax(1)
        correct += (clean_pred == y).sum().item()
        robust  += ((clean_pred == y) & (adv_pred == y)).sum().item()
        n += len(y)
    return dict(n=n, clean_correct=correct, pgd_robust=robust)

tf = transforms.ToTensor()
test_ds = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True, transform=tf)
rng = np.random.default_rng(1234)
idx = sorted(rng.choice(len(test_ds), size=50, replace=False).tolist())
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(test_ds, idx),
                                     batch_size=10, shuffle=False, num_workers=0)

drive_base_standard = Path('/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard')
drive_base_lipmargin = Path('/content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin')

CKPTS = {
    'standard': drive_base_standard / 'model.pt',
    'lipmargin': drive_base_lipmargin / 'model.pt',
}

for name, path in CKPTS.items():
    print(f'{name}: {path}  exists={path.exists()}')

EPS_LIST = [0.01, 0.03, 0.05, 0.1]

report = {}
for name, path in CKPTS.items():
    if not path.exists():
        print(f'[skip] {name}: {path}'); continue
    model = load_vit(path)
    print(f'\n══ {name} ══════════════════════════════')
    report[name] = {}
    for eps in EPS_LIST:
        t0 = time.time()
        r = eval_pgd(model, loader, eps)
        report[name][eps] = r
        print(f'  ε={eps:.3f}  clean={r["clean_correct"]}/{r["n"]}  '
              f'PGD-robust={r["pgd_robust"]}/{r["n"]}  ({time.time()-t0:.1f}s)')

out = Path('results/vit_p2'); out.mkdir(parents=True, exist_ok=True)
(out / 'pgd_report.json').write_text(json.dumps(report, indent=2))
print(f'\nSaved → {out / "pgd_report.json"}')

standard: /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_standard/model.pt  exists=True
lipmargin: /content/drive/My Drive/thesis-formal-verification/runs/vit_tiny_lipmargin/model.pt  exists=True

══ standard ══════════════════════════════
  ε=0.010  clean=48/50  PGD-robust=46/50  (16.2s)
  ε=0.030  clean=48/50  PGD-robust=32/50  (15.6s)
  ε=0.050  clean=48/50  PGD-robust=4/50  (15.6s)
  ε=0.100  clean=48/50  PGD-robust=0/50  (15.7s)

══ lipmargin ══════════════════════════════
  ε=0.010  clean=3/50  PGD-robust=0/50  (15.6s)
  ε=0.030  clean=3/50  PGD-robust=0/50  (15.4s)
  ε=0.050  clean=3/50  PGD-robust=0/50  (15.5s)
  ε=0.100  clean=3/50  PGD-robust=0/50  (15.5s)

Saved → results/vit_p2/pgd_report.json


In [ ]:
drive_out = Path('/content/drive/My Drive/thesis-formal-verification/results/vit_p2')
drive_out.mkdir(parents=True, exist_ok=True)
report_path = out / 'pgd_report.json'
(drive_out / 'pgd_report.json').write_text(report_path.read_text())
print(f'Saved drive copy → {drive_out / "pgd_report.json"}')